# MNIST demo with CREST


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Save the start time
import time
start_time = time.time()

# Allow crest to be imported
%cd -q ..

/Users/draghun1/miniconda3/envs/crest_cpu/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
from crest import DataServer

import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Dropout, Flatten, Dense, Lambda
from crest.model.HierarchalTensorGraph import HierarchalTensorGraph
from crest.model.KerasNode import KerasNode
from crest.model.TensorSpec import TensorSpec
from crest.utils.Metrics import Metrics


/Users/draghun1/miniconda3/envs/crest_cpu/lib/python3.10/site-packages/sklearn/utils/__init__.py:15: UserWarning: A NumPy version >=1.22.4 and <1.29.0 is required for this version of SciPy (detected version 1.22.3)
  from scipy.sparse import issparse


#### Load MNIST Datasets


In [4]:
# Load datasets from server
ds_train, ds_valid, ds_test, ds_pred = DataServer.load('mnist')

# Get input/output labels
[INP, OUT] = ds_train.labels

# Num classes
n_classes = 10
patch_size = 28

#### Create the HierarchalTensorGraph


In [5]:

layers = [
    KerasNode(Lambda(lambda x: tf.cast(x, tf.float32)), name='layer_0', inputs={INP: TensorSpec(
        [None, patch_size, patch_size])}, outputs={OUT: TensorSpec([None, 28, 28])}),        # Cast dtype
    KerasNode(Lambda(lambda x: tf.expand_dims(x, axis=-1)), name='layer_1', inputs={INP: TensorSpec(
        [None, patch_size, patch_size])}, outputs={OUT: TensorSpec([None, 28, 28, 1])}),  # Add channel dimension
    KerasNode(Conv2D(10, 3, padding='same', activation='relu'), name='layer_2', inputs={INP: TensorSpec(
        [None, 28, 28, 1])}, outputs={OUT: TensorSpec([None, 28, 28, 10])}),  # Convolutional layer
    KerasNode(Flatten(), name='layer_3', inputs={INP: TensorSpec(
        [None, 28, 28, 10])}, outputs={OUT: TensorSpec([None, 28*28*10])}),  # Flatten
    KerasNode(Dropout(0.3), name='layer_4', inputs={INP: TensorSpec(
        [None, 28*28*10])}, outputs={OUT: TensorSpec([None, 28*28*10])}),  # Dropout
    KerasNode(Dense(n_classes, activation='softmax'), name='layer_5', inputs={
              INP: TensorSpec([None, 28*28*10])}, outputs={OUT: TensorSpec([None, 10])})  # Output layer
]

for i in layers[:-1]:
    i.rename_io(outputs_map={OUT: INP})

graph = HierarchalTensorGraph(
    name='MNIST',
    inputs={INP: TensorSpec([None, patch_size, patch_size])},
    outputs={OUT: TensorSpec([None, 10])}
)

graph.add_edge('input', layers[0])

for i, layer in enumerate(layers[1:]):
    graph.add_edge(layers[i], layers[i+1])

graph.add_edge(layers[-1], 'output')

INFO:crest.model.HierarchalTensorGraph:Created HierarchalTensorGraph layer_0
INFO:crest.model.HierarchalTensorGraph:Created HierarchalTensorGraph layer_1
INFO:crest.model.HierarchalTensorGraph:Created HierarchalTensorGraph layer_2
INFO:crest.model.HierarchalTensorGraph:Created HierarchalTensorGraph layer_3
INFO:crest.model.HierarchalTensorGraph:Created HierarchalTensorGraph layer_4
INFO:crest.model.HierarchalTensorGraph:Created HierarchalTensorGraph layer_5
INFO:crest.model.HierarchalTensorGraph:Representing HTG layer_0
INFO:crest.model.HierarchalTensorGraph:Renaming inputs and outputs for node None
INFO:crest.model.HierarchalTensorGraph:Getting node None in HTG layer_0
INFO:crest.model.HierarchalTensorGraph:Representing HTG layer_1
INFO:crest.model.HierarchalTensorGraph:Renaming inputs and outputs for node None
INFO:crest.model.HierarchalTensorGraph:Getting node None in HTG layer_1
INFO:crest.model.HierarchalTensorGraph:Representing HTG layer_2
INFO:crest.model.HierarchalTensorGraph:R

#### Build a model using the HTG


In [6]:
from crest import Model

# Define a custom metric if desired
def custom_metric(y_true, y_pred):
    squared_difference = tf.square(y_true - y_pred)
    return tf.reduce_mean(squared_difference, axis=-1)

# Set build options
build_kwargs = {
    'optimizer': 'Adam',
    'loss': 'sparse_categorical_crossentropy',
    'metrics': ['sparse_categorical_accuracy', custom_metric]
}

# Create anad build a model using the HTG
model = Model(graph)

model.metric.register(event='custom_metric', callback=custom_metric)

model.build(**build_kwargs)

INFO:crest.model.Model:Initializing CREST Model
INFO:crest.model.HierarchalTensorGraph:Calling HTG MNIST
INFO:crest.model.HierarchalTensorGraph:Checking if HTG MNIST is a basenode
INFO:crest.model.HierarchalTensorGraph:Getting all sources in HTG MNIST
INFO:crest.model.HierarchalTensorGraph:Getting all sinks in HTG MNIST
INFO:crest.model.HierarchalTensorGraph:Representing HTG MNIST
INFO:crest.model.HierarchalTensorGraph:Applying feature map to input in HTG MNIST
INFO:crest.model.HierarchalTensorGraph:Checking if HTG MNIST is a basenode
INFO:crest.model.HierarchalTensorGraph:Getting all nodes in HTG MNIST
INFO:crest.model.HierarchalTensorGraph:Getting node input in HTG MNIST
INFO:crest.model.HierarchalTensorGraph:Getting node layer_0 in HTG MNIST
INFO:crest.model.HierarchalTensorGraph:Getting node layer_1 in HTG MNIST
INFO:crest.model.HierarchalTensorGraph:Getting node layer_2 in HTG MNIST
INFO:crest.model.HierarchalTensorGraph:Getting node layer_3 in HTG MNIST
INFO:crest.model.Hierarcha

#### Train the model


In [7]:
# Create the training and validation batchers
from crest.data.batching.Batcher import Batcher

batch_kwargs = {
    'batch_size': 200,            # 200 samples in each batch
    'features': [[INP], [OUT]],  # [{'image': ...}, {'class': ...}]
    'repeat': True,           # Yield indefinitely
    'workers': 0               # Number of background processes
}

batch_train = Batcher(ds_train, **batch_kwargs)
batch_kwargs['shuffle'] = False    # Turn shuffle off for prediction
batch_valid = Batcher(ds_valid,  **batch_kwargs)

with batch_train as train, batch_valid as valid:
    model.fit(train, **{
        'validation_data': valid,
        'steps_per_epoch': 250,  # 50k samples / 200 samples per batch
        'validation_steps': 50,  # 10k samples / 200 samples per batch
        'epochs': 5,
        'verbose': 1
    })

INFO:crest.model.Model:Training Keras Model
INFO:crest.model.Model:Called _make_batcher, making batcher from dataset
INFO:crest.model.Model:Called _make_batcher, making batcher from dataset
INFO:crest.model.Model:Called _make_batcher, making batcher from dataset


Epoch 1/5
  1/250 [..............................] - ETA: 39s - loss: 38.4667 - sparse_categorical_accuracy: 0.1100 - custom_metric: 27.9570

2025-06-03 11:07:50.217035: W tensorflow/core/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


250/250 [==============================] - 5s 18ms/step - loss: 1.9895 - sparse_categorical_accuracy: 0.8741 - custom_metric: 27.4550 - val_loss: 0.3452 - val_sparse_categorical_accuracy: 0.9630 - val_custom_metric: 27.4870
Epoch 2/5
250/250 [==============================] - 5s 19ms/step - loss: 0.3816 - sparse_categorical_accuracy: 0.9452 - custom_metric: 27.2785 - val_loss: 0.2575 - val_sparse_categorical_accuracy: 0.9730 - val_custom_metric: 27.4870
Epoch 3/5
250/250 [==============================] - 5s 19ms/step - loss: 0.2185 - sparse_categorical_accuracy: 0.9594 - custom_metric: 27.3646 - val_loss: 0.2461 - val_sparse_categorical_accuracy: 0.9760 - val_custom_metric: 27.4870
Epoch 4/5
250/250 [==============================] - 5s 19ms/step - loss: 0.1582 - sparse_categorical_accuracy: 0.9665 - custom_metric: 27.4638 - val_loss: 0.2126 - val_sparse_categorical_accuracy: 0.9760 - val_custom_metric: 27.4869
Epoch 5/5
250/250 [==============================] - 5s 19ms/step - loss: 

#### Evaluate the model


In [8]:
# Evaluate the model on the test set
batch_test = Batcher(ds_test, **batch_kwargs)
evaluate_kwargs = {
    'verbose': 1,
    'return_dict': True
}

with batch_test as batcher:
    res = model.evaluate(batcher, verbose=1, return_dict=1)

INFO:crest.model.Model:Evaluating Keras Model
INFO:crest.model.Model:Called _make_batcher, making batcher from dataset


1/1 [==============================] - 0s 12ms/step - loss: 0.0770 - sparse_categorical_accuracy: 0.9750 - custom_metric: 26.3272


Accuracy on the full test set should be > ~97%.


### Make predictions


In [9]:
# Create batcher for predictions
batch_pred = Batcher(ds_pred, **{  # Reset batch iterator
    'batch_size': 200,
    'features': [INP],
    'repeat': True,
    'workers': 0,
    'shuffle': False
})

# Set kwargs for Model.predict() (can be the same as Keras.predict)
pred_kwargs = {
    'batch_size': 200,
    'verbose': False,
    'steps': 10000//200
}

# Generate predictions
with batch_pred as batcher:
    model_pred_batch = model.predict(
        batch_pred, **pred_kwargs)  # Using a batcher
    model_pred_dataset = model.predict(
        ds_pred, **pred_kwargs)   # Using a dataset
    print('Difference:',
          (model_pred_batch['class'] - model_pred_dataset['class']).sum())

INFO:crest.model.Model:Predicting with Keras Model
INFO:crest.model.Model:Called _make_batcher, making batcher from dataset
INFO:crest.model.Model:Predicting with Keras Model
INFO:crest.model.Model:Called _make_batcher, making batcher from dataset
/Users/draghun1/miniconda3/envs/crest_cpu/lib/python3.10/site-packages/sklearn/utils/__init__.py:15: UserWarning: A NumPy version >=1.22.4 and <1.29.0 is required for this version of SciPy (detected version 1.22.3)
  from scipy.sparse import issparse
2025-06-03 11:08:19,985 |  43382 | MainThread |  655:Batcher.p.. | WARNING | Not enough sample blocks for workers! Set Batcher.duplicate=True.
/Users/draghun1/miniconda3/envs/crest_cpu/lib/python3.10/site-packages/sklearn/utils/__init__.py:15: UserWarning: A NumPy version >=1.22.4 and <1.29.0 is required for this version of SciPy (detected version 1.22.3)
  from scipy.sparse import issparse
2025-06-03 11:08:23,789 |  43403 | MainThread |  655:Batcher.p.. | WARNING | Not enough sample blocks for w

Difference: 0.0


#### Adding additional output (e.g., lat and lon) when making a predictions


In [10]:
# Create inputs with additional features lat and lon
import numpy as np
from crest.data.loading import StructuredDataset
x = ds_pred.data[0]
lat = np.arange(len(x), dtype=float)
lon = np.arange(len(x), dtype=float)
x_aux = (lat, lon, x)
ds_aux = StructuredDataset(*x_aux, labels=['lat', 'lon', INP])

# Specify additional output as coords when making predictions
model.predict(ds_aux, coords=['lat', 'lon'], **pred_kwargs)

INFO:crest.model.Model:Predicting with Keras Model
INFO:crest.model.Model:Called _make_batcher, making batcher from dataset
/Users/draghun1/miniconda3/envs/crest_cpu/lib/python3.10/site-packages/sklearn/utils/__init__.py:15: UserWarning: A NumPy version >=1.22.4 and <1.29.0 is required for this version of SciPy (detected version 1.22.3)
  from scipy.sparse import issparse
2025-06-03 11:08:29,629 |  43447 | MainThread |  655:Batcher.p.. | WARNING | Not enough sample blocks for workers! Set Batcher.duplicate=True.
/Users/draghun1/miniconda3/envs/crest_cpu/lib/python3.10/site-packages/sklearn/utils/__init__.py:15: UserWarning: A NumPy version >=1.22.4 and <1.29.0 is required for this version of SciPy (detected version 1.22.3)
  from scipy.sparse import issparse
2025-06-03 11:08:33,289 |  43462 | MainThread |  655:Batcher.p.. | WARNING | Not enough sample blocks for workers! Set Batcher.duplicate=True.


{'class': array([[2.5293883e-17, 2.6720998e-22, 2.5273281e-14, ..., 1.0000000e+00,
         1.5399720e-12, 1.1862227e-12],
        [1.2633188e-14, 1.6135287e-16, 1.0000000e+00, ..., 5.6015426e-29,
         9.4904359e-18, 1.8417926e-25],
        [7.6028568e-06, 9.9998748e-01, 5.3074011e-08, ..., 1.1172298e-11,
         1.4436833e-06, 6.9064313e-13],
        ...,
        [3.9816359e-19, 7.4953952e-17, 2.9840732e-21, ..., 7.2323254e-09,
         2.9467166e-09, 3.2954295e-09],
        [3.1119652e-14, 1.3868359e-18, 5.2922903e-17, ..., 2.7543069e-20,
         3.7836931e-10, 3.3425873e-15],
        [4.6782755e-17, 5.1439861e-16, 2.2967244e-11, ..., 2.8475179e-15,
         7.5307404e-15, 5.4071934e-20]], dtype=float32),
 'lat': array([0.000e+00, 1.000e+00, 2.000e+00, ..., 9.997e+03, 9.998e+03,
        9.999e+03]),
 'lon': array([0.000e+00, 1.000e+00, 2.000e+00, ..., 9.997e+03, 9.998e+03,
        9.999e+03])}

In [11]:
print('Elapsed time = ', time.time() - start_time, '[s]')

Elapsed time =  57.2433979511261 [s]


model save and load

In [12]:
model.save('crest_cache', save_metrics=True)

INFO:crest.model.Model:Save Keras Model
INFO:crest.model.HierarchalTensorGraph:Serializing HTG MNIST to JSON
INFO:crest.model.HierarchalTensorGraph:Checking if HTG MNIST is a basenode
INFO:crest.model.HierarchalTensorGraph:Getting all edges in HTG MNIST
INFO:crest.model.HierarchalTensorGraph:Getting all nodes in HTG MNIST
INFO:crest.model.Model:Save model as custom_model
INFO:crest.model.Model:Saving model with tensorflow keras.


INFO:tensorflow:Assets written to: crest_cache/assets


INFO:tensorflow:Assets written to: crest_cache/assets


['custom_metric']
['unbiased_rmse', 'fluctuation_complexity', 'metric_entropy', 'relative_error', 'triple_collocation_error', 'nse', 'nse_log', 'mse', 'kge', 'lkg', 'pearson', 'alpha_nse', 'beta_nse', 'custom_metric']
{"custom_metric": {"py/function": "__main__.custom_metric"}}


In [13]:
loaded_model = model.load('crest_cache', 'keras', True)

INFO:crest.model.HierarchalTensorGraph:Created HierarchalTensorGraph MNIST
INFO:crest.model.HierarchalTensorGraph:Representing HTG MNIST
INFO:crest.model.HierarchalTensorGraph:Created HierarchalTensorGraph layer_0
INFO:crest.model.HierarchalTensorGraph:Representing HTG layer_0
INFO:crest.model.HierarchalTensorGraph:Adding node HierarchalTensorGraph("layer_0", id=14333060160) to HTG MNIST
INFO:crest.model.HierarchalTensorGraph:Representing HTG MNIST
INFO:crest.model.HierarchalTensorGraph:Representing HTG layer_0
INFO:crest.model.HierarchalTensorGraph:Getting node HierarchalTensorGraph("layer_0", id=14333060160) in HTG MNIST
INFO:crest.model.HierarchalTensorGraph:Getting all nodes in HTG MNIST
INFO:crest.model.HierarchalTensorGraph:Iterating through all nodes in HTG MNIST
INFO:crest.model.HierarchalTensorGraph:Getting all nodes in HTG MNIST
INFO:crest.model.HierarchalTensorGraph:Iterating through all nodes in HTG MNIST
INFO:crest.model.HierarchalTensorGraph:Checking if HTG layer_0 is a b

True


In [14]:
# check that loaded model performs the same as original
result_loaded = loaded_model.predict(ds_aux, coords=['lat', 'lon'], **pred_kwargs)
result = model.predict(ds_aux, coords=['lat', 'lon'], **pred_kwargs)

print('Difference:', (result['class'] - result_loaded['class']).sum())

INFO:crest.model.Model:Predicting with Keras Model
INFO:crest.model.Model:Called _make_batcher, making batcher from dataset
/Users/draghun1/miniconda3/envs/crest_cpu/lib/python3.10/site-packages/sklearn/utils/__init__.py:15: UserWarning: A NumPy version >=1.22.4 and <1.29.0 is required for this version of SciPy (detected version 1.22.3)
  from scipy.sparse import issparse
2025-06-03 11:08:39,621 |  43506 | MainThread |  655:Batcher.p.. | WARNING | Not enough sample blocks for workers! Set Batcher.duplicate=True.
/Users/draghun1/miniconda3/envs/crest_cpu/lib/python3.10/site-packages/sklearn/utils/__init__.py:15: UserWarning: A NumPy version >=1.22.4 and <1.29.0 is required for this version of SciPy (detected version 1.22.3)
  from scipy.sparse import issparse
2025-06-03 11:08:43,382 |  43517 | MainThread |  655:Batcher.p.. | WARNING | Not enough sample blocks for workers! Set Batcher.duplicate=True.
INFO:crest.model.Model:Predicting with Keras Model
INFO:crest.model.Model:Called _make_b

Difference: 0.0
